# MATH GR5360 Final Project — Master Pipeline

One-click amalgamated notebook for the project-facing trend-following workflow. It imports the shared engine, runs the diagnostics layer to locate the time-scale of the inefficiency, executes the trend-following walk-forward engine, and computes the OOS and full-sample TF metrics.


In [ ]:
MARKET_SELECT = 'TY'
QUICK_TEST = True
WALKFORWARD_MODE = 'tf'
RUN_EXTENDED_SURFACE = False


In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'mafn_engine').exists() else CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mafn_engine import (
    COLUMBIA_CORE,
    COLUMBIA_NAVY,
    COLUMBIA_RED,
    apply_columbia_theme,
    choose_tf_story_configuration,
    default_tf_grid,
    get_market,
    load_ohlc,
    performance_from_ledger,
    prepare_analysis_frame,
    run_backtest,
    run_diagnostics,
    validate_ohlc,
    walk_forward,
    walk_forward_surface,
)

apply_columbia_theme()
MARKET = get_market(MARKET_SELECT)
DATA_DIR = str(PROJECT_ROOT / 'data')
print(f"Market: {MARKET.ticker} - {MARKET.name} ({MARKET.exchange})")
print(f"PV=${MARKET.PV:,}  Slippage=${MARKET.slpg}  E0=${MARKET.E0:,.0f}")


In [ ]:
DEFAULT_T_VALUES = [1, 2, 3, 4, 5, 6]
DEFAULT_TAU_VALUES = [1, 2, 3, 4]


def _grid_signature(grid: dict[str, np.ndarray]) -> tuple[tuple[str, tuple[float, ...]], ...]:
    signature = []
    for key in sorted(grid):
        values = []
        for value in np.asarray(grid[key]).tolist():
            if isinstance(value, (int, np.integer)):
                values.append(int(value))
            else:
                values.append(float(value))
        signature.append((key, tuple(values)))
    return tuple(signature)


def ensure_analysis_state(force: bool = False) -> None:
    global MARKET, full_df, validation, analysis_df, tf_grid, _analysis_market

    MARKET = get_market(MARKET_SELECT)
    if force or globals().get('_analysis_market') != MARKET_SELECT or 'analysis_df' not in globals():
        full_df = load_ohlc(DATA_DIR, MARKET_SELECT, fallback_synthetic=False)
        validation = validate_ohlc(full_df)
        analysis_df = prepare_analysis_frame(full_df, MARKET_SELECT)
        _analysis_market = MARKET_SELECT

    tf_grid = default_tf_grid(MARKET_SELECT, quick=QUICK_TEST)


def ensure_diagnostics_state(force: bool = False) -> None:
    global diagnostics_bundle, vr_price_df, regime_table, trend_profile, _diagnostics_market

    ensure_analysis_state(force=force)
    if force or globals().get('_diagnostics_market') != MARKET_SELECT or 'diagnostics_bundle' not in globals():
        diagnostics_bundle = run_diagnostics(analysis_df, MARKET_SELECT)
        vr_price_df = diagnostics_bundle['vr_price_df']
        regime_table = diagnostics_bundle['regime_table']
        trend_profile = diagnostics_bundle['trend_profile']
        _diagnostics_market = MARKET_SELECT


def ensure_walkforward_state(force: bool = False) -> None:
    global wf_bundle, wf_params, wf_equity, wf_ledger, wf_results, _wf_signature

    ensure_analysis_state(force=force)
    signature = (
        MARKET_SELECT,
        WALKFORWARD_MODE,
        bool(QUICK_TEST),
        4,
        1,
        _grid_signature(tf_grid),
    )

    if force or globals().get('_wf_signature') != signature or 'wf_bundle' not in globals():
        wf_bundle = walk_forward(
            analysis_df,
            MARKET_SELECT,
            mode=WALKFORWARD_MODE,
            tf_grid=tf_grid,
            T_years=4,
            tau_quarters=1,
            quick=QUICK_TEST,
            verbose=True,
        )
        _wf_signature = signature

    wf_params = wf_bundle['params']
    wf_equity = wf_bundle['equity']
    wf_ledger = wf_bundle['ledger']
    wf_results = wf_params


def _tf_full_sample_config() -> dict[str, object]:
    return choose_tf_story_configuration(MARKET_SELECT, tf_grid=tf_grid, params_df=wf_params)


def ensure_full_sample_state(force: bool = False) -> None:
    global modal_cfg, full_sample_result, oos_metrics, full_sample_metrics, _full_sample_signature

    ensure_walkforward_state(force=force)
    ensure_diagnostics_state(force=force)
    candidate = _tf_full_sample_config()

    signature = (MARKET_SELECT, tuple(sorted(candidate.items())))
    if force or globals().get('_full_sample_signature') != signature or 'full_sample_result' not in globals():
        modal_cfg = dict(candidate)
        full_sample_result = run_backtest(
            analysis_df,
            MARKET_SELECT,
            modal_cfg['family'],
            {k: v for k, v in modal_cfg.items() if k != 'family'},
        )
        oos_metrics = performance_from_ledger(
            wf_ledger,
            wf_equity['OOS_Equity'].values if len(wf_equity) else np.array([MARKET.E0]),
            MARKET_SELECT,
        )
        full_sample_metrics = performance_from_ledger(
            full_sample_result['Ledger'],
            full_sample_result['Equity'],
            MARKET_SELECT,
        )
        _full_sample_signature = signature


ensure_diagnostics_state(force=True)
print(validation)
print('Diagnostics complete. The strategy workflow below is fixed to trend-following only.')
print(trend_profile['narrative'])
regime_table.head()


In [ ]:
ensure_full_sample_state(force=True)

print()
print('MASTER SUMMARY')
print('-' * 72)
print(f"Periods: {len(wf_params)}  |  OOS ledger rows: {len(wf_ledger)}")
print(f"TF config: {modal_cfg}")
pd.DataFrame([oos_metrics, full_sample_metrics], index=['OOS', 'Full Sample'])


In [ ]:
ensure_full_sample_state()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].bar(range(len(vr_price_df)), vr_price_df['VR'] - 1.0, color=COLUMBIA_CORE, edgecolor=COLUMBIA_NAVY)
axes[0, 0].axhline(0.0, color=COLUMBIA_RED, linestyle='--', linewidth=1)
axes[0, 0].set_xticks(range(len(vr_price_df)))
axes[0, 0].set_xticklabels(vr_price_df['time_scale'], rotation=45, ha='right')
axes[0, 0].set_title('Variance Ratio on Signed Price Changes')

if len(wf_params) and 'L' in wf_params:
    axes[0, 1].plot(wf_params['Period'], wf_params['L'], marker='o', color=COLUMBIA_NAVY)
    axes[0, 1].set_title('Selected TF Lookback by Period')
    axes[0, 1].set_xlabel('Period')
    axes[0, 1].set_ylabel('L (bars)')
else:
    axes[0, 1].text(0.5, 0.5, 'No TF parameter history generated', ha='center', va='center')
    axes[0, 1].set_axis_off()

if len(wf_equity):
    axes[1, 0].plot(wf_equity.index, wf_equity['OOS_Equity'], color=COLUMBIA_NAVY)
    axes[1, 0].set_title('Concatenated OOS Equity')
    axes[1, 0].set_ylabel('Equity ($)')
else:
    axes[1, 0].text(0.5, 0.5, 'No OOS equity generated', ha='center', va='center')
    axes[1, 0].set_axis_off()

axes[1, 1].plot(analysis_df.index, full_sample_result['Equity'], color=COLUMBIA_CORE)
axes[1, 1].set_title('Full-Sample TF Equity')
axes[1, 1].set_ylabel('Equity ($)')

plt.tight_layout()
plt.show()


In [ ]:
if RUN_EXTENDED_SURFACE:
    ensure_analysis_state()
    surface_df = walk_forward_surface(
        analysis_df,
        MARKET_SELECT,
        mode=WALKFORWARD_MODE,
        tf_grid=tf_grid,
        T_values=DEFAULT_T_VALUES,
        tau_values=DEFAULT_TAU_VALUES,
        quick=QUICK_TEST,
        verbose=False,
    )
    surface_valid = surface_df[~surface_df['error']].copy()
    surface_valid
